In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. THIẾT LẬP CƠ BẢN
train_dir = "/content/drive/MyDrive/Colab Notebooks/Món ăn VN"
img_width, img_height = 200, 200
batch_size = 40

# 2. TĂNG CƯỜNG DỮ LIỆU (KHÓA HUẤN LUYỆN KHẮC NGHIỆT)
datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=30,        # Xoay ảnh 30 độ
    width_shift_range=0.2,    # Dịch chuyển ngang
    height_shift_range=0.2,   # Dịch chuyển dọc
    shear_range=0.2,          # Làm méo ảnh
    zoom_range=0.2,           # Phóng to/thu nhỏ ngẫu nhiên
    brightness_range=[0.7, 1.3], # <--- VŨ KHÍ MỚI: Giả lập ảnh thiếu sáng và chói sáng
    horizontal_flip=True,     # Lật ngang
    fill_mode="nearest",
    validation_split=0.2      # Dành 20% test
)

# 3. TẠO BỘ DỮ LIỆU
train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode="categorical",
    subset="training"
)

validation_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode="categorical",
    subset="validation"
)

num_classes = train_generator.num_classes
class_labels = {v: k for k, v in train_generator.class_indices.items()}
print(f"Đã sẵn sàng nhận diện {num_classes} món ăn!")

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Vẫn giữ mức phạt nặng để chặn đứng ý định học vẹt
l2_reg = regularizers.l2(0.001)

model = Sequential([
    # Khối 1: Lọc đặc trưng cơ bản (Chỉ dùng 16 bộ lọc thay vì 32)
    Conv2D(16, (3,3), padding='same', activation="relu", kernel_regularizer=l2_reg, input_shape=(img_width, img_height, 3)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    # Khối 2: Lọc đặc trưng trung bình (Chỉ dùng 32 bộ lọc)
    Conv2D(32, (3,3), padding='same', activation="relu", kernel_regularizer=l2_reg),
    BatchNormalization(),
    MaxPooling2D(2,2),

    # Khối 3: Chốt sổ đặc trưng (Dừng lại ở 64, tuyệt đối không lên 128 hay 256)
    Conv2D(64, (3,3), padding='same', activation="relu", kernel_regularizer=l2_reg),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.4),


    Conv2D(128, (3,3), padding='same', activation="relu", kernel_regularizer=l2_reg),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.4),


    Conv2D(256, (3,3), padding='same', activation="relu", kernel_regularizer=l2_reg),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.4),
    # Bóp phẳng dữ liệu bằng GAP (Vũ khí tối thượng cho dữ liệu nhỏ)
    GlobalAveragePooling2D(),

    # Lớp suy luận (Chỉ cần 64 nơ-ron là quá đủ cho 300 ảnh)
    Dense(256, activation="relu", kernel_regularizer=l2_reg),
    BatchNormalization(),
    Dropout(0.5),

    # Đầu ra phân loại (num_classes lúc này sẽ tự động là 6)
    Dense(num_classes, activation="softmax")
])

# Tốc độ học khuyên dùng cho dữ liệu nhỏ: Chậm mà chắc
model.compile(optimizer=Adam(learning_rate=0.001), loss="categorical_crossentropy", metrics=["accuracy"])
early_stopping = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1)

history = model.fit(
    train_generator,
    epochs=100,
    validation_data=validation_generator,
    callbacks=[early_stopping, reduce_lr]
)

# In biểu đồ
plt.figure(figsize=(10, 5))
plt.plot(history.history['accuracy'], label="Độ chính xác Train")
plt.plot(history.history['val_accuracy'], label="Độ chính xác Test (Val)")
plt.title("Biểu đồ độ chính xác theo số lần học")
plt.xlabel("Số lần học (Epochs)")
plt.ylabel("Độ chính xác")
plt.legend()
plt.show()

In [ ]:
from google.colab import drive
# Kết nối Colab với Google Drive của bạn (sẽ có popup yêu cầu cấp quyền)
drive.mount('/content/drive')

# Lưu thẳng mô hình vào thư mục gốc của Google Drive
model.save('/content/drive/MyDrive/Nhận dạng món ăn.keras')

print("Đã lưu mô hình vĩnh viễn vào Google Drive!")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ==========================================
# 1. KHÔI PHỤC MÔ HÌNH & TỰ ĐỘNG LẤY NHÃN
# ==========================================
# Tải lại mô hình CNN "trâu bò" của bạn
model = load_model("/content/drive/MyDrive/Nhận dạng món ăn.keras")

# Khai báo lại đường dẫn thư mục gốc chứa ảnh (đảm bảo Google Drive đã được kết nối)
train_dir = "/content/drive/MyDrive/Colab Notebooks/Món ăn VN"

# DÙNG MẸO: Gọi ImageDataGenerator một cách đơn giản nhất chỉ để "quét" lấy tên thư mục
dummy_datagen = ImageDataGenerator()
dummy_generator = dummy_datagen.flow_from_directory(
    train_dir,
    batch_size=1, # Cho thông số nhỏ nhất để chạy chớp nhoáng
    class_mode="categorical"
)

# Đảo ngược dictionary để tra cứu kết quả (ví dụ: {0: 'Nguoi_A', 1: 'Nguoi_B'})
class_labels = {v: k for k, v in dummy_generator.class_indices.items()}


# ==========================================
# 2. HÀM CĂN CHỈNH ẢNH (Giữ nguyên tỷ lệ)
# ==========================================
def pad_and_resize(image, target_size=200):
    h, w = image.shape[:2]
    max_dim = max(h, w)

    top = (max_dim - h) // 2
    bottom = max_dim - h - top
    left = (max_dim - w) // 2
    right = max_dim - w - left

    square_img = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT, value=[0, 0, 0])
    resized_img = cv2.resize(square_img, (target_size, target_size))
    return resized_img


# ==========================================
# 3. ĐƯA ẢNH VÀO DỰ ĐOÁN
# ==========================================
# Thay bằng đường dẫn bức ảnh thực tế bạn muốn test
img_path = "/content/sddefault (1).jpg"
img = cv2.imread(img_path)

if img is not None:
    # Chuyển BGR -> RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Căn chỉnh ảnh an toàn
    img_aligned = pad_and_resize(img_rgb, target_size=200)

    # Chuẩn hóa
    img_ready = img_aligned.astype('float32') / 255.0
    img_ready = np.expand_dims(img_ready, axis=0)

    # Mô hình dự đoán
    preds = model.predict(img_ready)
    class_idx = np.argmax(preds)

    # Đọc kết quả
    predicted_label = class_labels[class_idx]
    confidence = preds[0][class_idx] * 100

    # Hiển thị
    plt.figure(figsize=(6, 6))
    plt.imshow(img_aligned)
    plt.title(f"Dự đoán: {predicted_label}\nĐộ tự tin: {confidence:.2f}%", fontsize=14, color='green', fontweight='bold')
    plt.axis('off')
    plt.show()
else:
    print(f"Lỗi: Không đọc được ảnh tại: {img_path}. Nhớ kiểm tra lại tên file nhé!")